# Demo 3: A booster, day by day

A patient is sick. **Every sick day** the doctor decides again: give a booster today, yes or no?

- Every sick day costs points, and a booster costs extra.
- A booster **changes the chances** for tomorrow: more likely to recover, less likely to die.
- The longer someone is sick, the more likely they recover **by themselves**.

So is a booster worth it on day 1? And on day 6? The cost comes today, the benefit comes later: the classic RL question.

| RL term | Here |
|---|---|
| **State** | sick on day 1, 2, ... 7 (day 7 also means "longer"), recovered or dead |
| **Action** | booster or no booster, every sick day |
| **Transition** | the chance of tomorrow being sick, recovered or dead: it depends on the day and on the booster |
| **Reward** | minus points for every sick day, for the booster, and for a death |
| **Episode** | one patient, from the first sick day until recovered or dead |

*The chances and costs are invented for teaching. This is not medical advice.*

**How to use:** click *Run → Run All Cells*. To change something, edit a cell in part 1 or 2, run it, and run the cells from there onwards.

## 1. The problem

### The chances, per sick day

Without a booster. Whoever does not recover or die is sick again the next day.

In [ ]:
#             day:  1     2     3     4     5     6     7+
RECOVER =        [0.05, 0.08, 0.12, 0.18, 0.26, 0.36, 0.48]   # chance to recover by tomorrow
DIE =            [0.10, 0.08, 0.06, 0.04, 0.02, 0.01, 0.01]   # chance to die by tomorrow

# What a booster on that day changes
BOOSTER_RECOVER = +0.35   # extra chance to recover
BOOSTER_DIE = -0.04       # less chance to die (never below 0)

### Rewards

In [ ]:
SICK_DAY = -10        # every sick day
BOOSTER_PRICE = -50   # extra on a day with a booster
DEATH = -1000         # once, when the patient dies

### Turn this into states, actions and transitions

You do not need to change this cell: it writes out the tables the robot learns from. Run it after every change above.

In [ ]:
DAYS = len(RECOVER)
SICK = [f'sick day {day}' for day in range(1, DAYS + 1)]

STATES = SICK + ['recovered', 'dead']
END_STATES = ['recovered', 'dead']
START = {'sick day 1': 1.0}
ACTIONS = {state: ['no booster', 'booster'] for state in SICK}
REWARDS = {state: {'no booster': SICK_DAY, 'booster': SICK_DAY + BOOSTER_PRICE} for state in SICK}
LANDING_REWARDS = {'dead': DEATH}

TRANSITIONS = {}
for day, state in enumerate(SICK):
    tomorrow = SICK[min(day + 1, DAYS - 1)]             # after day 7 you stay at "day 7+"
    TRANSITIONS[state] = {}
    for action, extra_recover, extra_die in [('no booster', 0, 0), ('booster', BOOSTER_RECOVER, BOOSTER_DIE)]:
        recover = RECOVER[day] + extra_recover
        die = max(0, DIE[day] + extra_die)
        TRANSITIONS[state][action] = {'recovered': recover, 'dead': die, tomorrow: 1 - recover - die}

## 2. How the robot learns

| Setting | Low | High |
|---|---|---|
| **Learning rate** | slow but stable | fast but jumpy |
| **Exploration** | may never find out that a booster helps | keeps giving random boosters |
| **Discount** | only today counts | the coming days count too |

In [ ]:
LEARNING_RATE = 0.002  # above 0, up to 1 (low, because one death is a big shock)
EXPLORATION = 0.2      # how often it tries a random action
DISCOUNT = 0.95        # 0 up to, not including, 1: how much tomorrow counts
MAX_STEPS = 60         # longest patient story in days
SEED = 0               # change it to see whether the result was luck

## Switch on the robot

Run this cell once. The learning, checking and drawing happen behind the scenes, there is nothing to edit here.

In [ ]:
from rl_demos.build_your_own import *

connect(globals(), table_name='brain')    # lets the robot read your cells

In [ ]:
show_problem()      # all transition probabilities, and any mistakes such as chances that do not add up to 1

## 3. Train

First only 20 patients: the robot has barely tried anything.

In [ ]:
robot = new_robot()
train(robot, episodes=20)
show_policy(robot)

Now 50,000 patients (a few seconds). Run the cell again to train further.

In [ ]:
train(robot, episodes=50_000)
show_policy(robot)
show_learning({'robot': robot})

## 4. Follow one patient

Day by day with the learned policy. Run it again for another patient.

In [ ]:
watch(robot, steps=30, step_name='day')

## 5. Better than a simple rule?

In [ ]:
compare({'robot': robot, 'never a booster': 'no booster', 'always a booster': 'booster'})

## Discussion

- The robot gives boosters early and stops later. Nobody told it that. Where did it learn it from?
- Why is a booster on day 6 a waste, even though it raises the chance of recovering just as much as on day 1? (Look at `DIE` too: on day 6 there is little risk left to take away.)
- In part 5, "always a booster" scores almost as well as the robot. Why does stopping on day 5 save so little? (How many patients are still sick on day 5?)
- Who decides how many points a death "costs"? What happens if `DEATH = -100`, and is that what we want?
- The robot learned by giving random boosters to simulated patients. Why could you not do that in a real hospital?
- The state is "which sick day". What else would a real doctor want to know? (How sick, how many boosters so far, age, ...) What happens to the robot's brain then?